In [ ]:
import importlib.metadata
import os
import subprocess
import sys
import tempfile
from pathlib import Path

if "tensorflow" in sys.modules and os.environ.get("TF_USE_LEGACY_KERAS") != "1":
    raise RuntimeError(
        "Restart the Colab session, then run this notebook from its first cell. "
        "NIF requires the legacy Keras setting before TensorFlow is imported."
    )
os.environ["TF_USE_LEGACY_KERAS"] = "1"

COLAB_REQUIREMENTS = [
    "tensorflow==2.21.0",
    "tf-keras==2.21.0",
    "keras==3.13.2",
    "tensorflow-probability==0.25.0",
    "tensorflow-model-optimization==0.8.1",
    "numpy==2.1.3",
    "scipy==1.16.3",
    "matplotlib==3.10.0",
    "tensorboard==2.21.0",
    "absl-py==1.4.0",
]
NIF_COMMIT = "ad9113df14590189b12068b6b3fa7c8c8d9187d4"

if sys.version_info < (3, 10) or sys.version_info >= (3, 14):
    raise RuntimeError(
        "Use a Colab Python 3.10–3.13 runtime for this pinned environment."
    )

missing = []
for requirement in COLAB_REQUIREMENTS:
    package, expected = requirement.split("==")
    try:
        installed = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != expected:
        missing.append(requirement)
if missing:
    package_modules = {
        "tensorflow": "tensorflow",
        "tf-keras": "tf_keras",
        "keras": "keras",
        "numpy": "numpy",
        "scipy": "scipy",
        "matplotlib": "matplotlib",
        "tensorflow-probability": "tensorflow_probability",
        "tensorflow-model-optimization": "tensorflow_model_optimization",
        "tensorboard": "tensorboard",
        "absl-py": "absl",
    }
    changing = {item.split("==")[0] for item in missing}
    loaded = [
        module
        for package, module in package_modules.items()
        if package in changing and module in sys.modules
    ]
    if loaded:
        raise RuntimeError(
            "Restart the Colab session before changing packages; already imported: "
            + ", ".join(loaded)
        )
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--disable-pip-version-check",
            *COLAB_REQUIREMENTS,
        ]
    )

runtime_root = Path(tempfile.gettempdir()) / ("nif-tutorial1-" + NIF_COMMIT[:12])
stamp = runtime_root / ".nif-source-commit"
if not stamp.exists() or stamp.read_text().strip() != NIF_COMMIT:
    runtime_root.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--no-deps",
            "--upgrade",
            "--disable-pip-version-check",
            "--target",
            str(runtime_root),
            "git+https://github.com/pswpswpsw/nif.git@" + NIF_COMMIT,
        ]
    )
    stamp.write_text(NIF_COMMIT)


def replace_compatibility(path, old, new):
    content = path.read_text(encoding="utf-8")
    if old in content:
        path.write_text(content.replace(old, new), encoding="utf-8")
    elif new not in content:
        raise RuntimeError("Pinned NIF source changed unexpectedly: " + str(path))


optimizer_file = runtime_root / "nif" / "optimizers" / "external_optimizers.py"
replace_compatibility(
    optimizer_file,
    "from tensorflow.python.distribute import distribution_strategy_context as distribute_ctx",
    "from tensorflow.python.distribute import distribute_lib as distribute_ctx",
)
for optimizer_class in ("AdaBeliefOptimizer", "L4Adam"):
    replace_compatibility(
        optimizer_file,
        "class " + optimizer_class + "(tf.keras.optimizers.Optimizer):",
        "class " + optimizer_class + "(tf.keras.optimizers.legacy.Optimizer):",
    )
for name in ("lbfgs.py", "lbfgs_V2.py"):
    path = runtime_root / "nif" / "optimizers" / name
    content = path.read_text(encoding="utf-8")
    if "np.product(" in content:
        path.write_text(content.replace("np.product(", "np.prod("), encoding="utf-8")

if str(runtime_root) not in sys.path:
    sys.path.insert(0, str(runtime_root))
if "nif" in sys.modules:
    loaded_nif = Path(sys.modules["nif"].__file__).resolve()
    if not loaded_nif.is_relative_to(runtime_root.resolve()):
        raise RuntimeError(
            "Restart the session: an unrelated NIF installation is already imported."
        )
print("Python:", sys.version.split()[0])
for requirement in COLAB_REQUIREMENTS:
    package = requirement.split("==")[0]
    print(package + ":", importlib.metadata.version(package))
print("NIF source commit:", NIF_COMMIT)
print("Compatibility copy:", runtime_root / "nif")


In [ ]:
import tensorflow as tf
import nif
import numpy as np
import time
import logging
import contextlib
from matplotlib import pyplot as plt
from mpl_toolkits import mplot3d
from nif.optimizers import centralized_gradients_for_optimizer

# Generate 1D travelling wave data

In [ ]:
NT=10 # 20
NX=200

x = np.linspace(0,1,NX,endpoint=False)
t = np.linspace(0,100,NT,endpoint=False)

xx,tt=np.meshgrid(x,t)

omega = 4
c = 0.12/20
x0 = 0.2

u = np.exp(-1000*(xx-x0-c*tt)**2)*np.sin(omega*(xx-x0-c*tt))

# vis
plt.figure()
for i in range(NT):
    plt.plot(x,u[i,:],'-',label=str(i) + '-th time')

plt.xlabel('$x$',fontsize=25)
plt.ylabel('$u$',fontsize=25)

# vis iso
plt.figure(figsize=(4,4))
ax = plt.axes(projection='3d')
ax.plot_surface(xx,tt,u,cmap="rainbow", lw=2)#,rstride=1, cstride=1)
ax.view_init(57, -80)
ax.set_xlabel(r'$x$',fontsize=25)
ax.set_ylabel(r'$t$',fontsize=25)
ax.set_zlabel(r'$u$',fontsize=25)

plt.tight_layout()

In [ ]:
import shutil, subprocess
if shutil.which('nvidia-smi'):
    subprocess.run(['nvidia-smi', '-L'], check=True)
else:
    print('No NVIDIA GPU attached; this tutorial also runs on CPU.')


# Fitting $u(x,t) \approx u_{nif}(x,t) = u_{shape}(x; \Theta_{parameter}(t))$ 
where $u_{shape}$ and $\Theta_{parameter}$ are two feedforward neural networks



* We define a shape-net with 1 input $x$, 1 output $u$, 2 hidden layers, Swish activation function, each layer has 30 units

  * note that here we assume parameter-net fully parameterizes the shape-net, so `connectivity` is `full`

* Then we define a parameter-net, still 1 input, $t$, and 1 latent dimension, 2 layers with each layer 30 units, also activated by Swish



In [ ]:
cfg_shape_net = {
    "connectivity": 'full',
    "input_dim": 1,
    "output_dim": 1,
    "units": 30,
    "nlayers": 2,
    "activation": 'swish'
}
cfg_parameter_net = {
    "input_dim": 1,
    "latent_dim": 1,
    "units": 30,
    "nlayers": 2,
    "activation": 'swish',
}

* we can setup whether to enable
  * **multiple gpus**,
  * **mixed precision (fp32 for store and fp16 for computing)**

* However at Colab, we don't have multiple-gpu (unless we pay). So it won't be effective here. 

* But you can play around with mixed precision, if your GPU supports it. Typically I found there is a slight increase in training speed. 

In [ ]:
enable_multi_gpu = False
enable_mixed_precision = False

* now we can further setup 
  * the total number of epochs
  * initial learning rate
  * batch size
  * number of epochs for check point saving
  * number of epochs for logging error file
  * number of epochs for plotting figures

In [ ]:
nepoch = 5000
lr = 5e-3
batch_size = 512
checkpt_epoch = 1000
display_epoch = 100
print_figure_epoch = 100

* now let's read the data from the disk
* then we create a `tf.data.Dataset` with `features` as `t,x` and `target` as `u`. everything is point-wise.
* we will further shuffle, batching, and pre-fetch it so as to optimize the speed, in which case, we won't have data I/O as out bottleneck 

In [ ]:
# alternatively, you can get training data set from nif.demo for this example
from nif.demo import TravelingWave
tw = TravelingWave()
train_data = tw.data

num_total_data = train_data.shape[0]
train_dataset = tf.data.Dataset.from_tensor_slices((train_data[:, :2], train_data[:, -1:]))
train_dataset = train_dataset.shuffle(num_total_data).batch(batch_size).prefetch(tf.data.experimental.AUTOTUNE)



* Now, let's do a few checking on the options for `mixed_precision`

In [ ]:
# mixed precision?
if enable_mixed_precision:
    mixed_policy = "mixed_float16"
    # we might need this for `model.fit` to automatically do loss scaling
    policy = nif.mixed_precision.Policy(mixed_policy)
    nif.mixed_precision.set_global_policy(policy)
else:
    mixed_policy = 'float32'

* Then, we define a `Callback` for better handling those plotting issues in `Keras`

  * remember that this is how you will modify the output of the training and details on the plotting. 
  * so here we will plot training loss for every 100 epochs, as `loss.png`.
  * and we will also plot the contourf of `(t,x)-u` saved as `vis.png`
  * finally, I will save the weight in a `checkpoint`, which means we are not saving the whole model but just the checkpoint in `saved_weights/`. 

In [ ]:
class LossAndErrorPrintingCallback(tf.keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.train_begin_time = time.time()
        self.history_loss = []
        logging.basicConfig(filename='./log', level=logging.INFO, format='%(message)s')

    def on_epoch_begin(self, epoch, logs=None):
        self.ts = time.time()

    def on_epoch_end(self, epoch, logs=None):
        if epoch % display_epoch == 0:
            tnow = time.time()
            te = tnow - self.ts
            logging.info("Epoch {:6d}: avg.loss pe = {:4.3e}, {:d} points/sec, time elapsed = {:4.3f} hours".format(
                epoch, logs['loss'], int(batch_size / te), (tnow - self.train_begin_time) / 3600.0))
            self.history_loss.append(logs['loss'])
        if epoch % print_figure_epoch == 0:
            plt.figure()
            plt.semilogy(self.history_loss)
            plt.xlabel('epoch: per {} epochs'.format(print_figure_epoch))
            plt.ylabel('MSE loss')
            plt.savefig('./loss.png')
            plt.close()

            u_pred = self.model.predict(train_data[:,0:2]).reshape(10,200)
            fig,axs=plt.subplots(1,3,figsize=(16,4))
            im1=axs[0].contourf(tt, xx, train_data[:,-1].reshape(10,200),vmin=-5,vmax=5,levels=50,cmap='seismic')
            plt.colorbar(im1,ax=axs[0])

            im2=axs[1].contourf(tt, xx, u_pred,vmin=-5,vmax=5,levels=50,cmap='seismic')
            plt.colorbar(im2,ax=axs[1])

            im3=axs[2].contourf(tt, xx, (u_pred-train_data[:,-1].reshape(10,200)),vmin=-5,vmax=5,levels=50,cmap='seismic')
            plt.colorbar(im3,ax=axs[2])

            axs[0].set_xlabel('t')
            axs[0].set_ylabel('x')
            axs[0].set_title('true')
            axs[1].set_title('pred')
            axs[2].set_title('error')
            plt.savefig('vis.png')
            plt.close()

        if epoch % checkpt_epoch == 0 or epoch == nepoch - 1:
            print('save checkpoint epoch: %d...' % epoch)
            self.model.save_weights("./saved_weights/ckpt-{}/ckpt".format(epoch))

* Here we define another `Callback` to modulate the learning rate, it is intuitive and heuristic, so you need to learn how to tune it a bit. but most of the time it is okay. indeed, in our nif paper, we didn't do any tuning but here I am doing this just to show that we can. 

In [ ]:
def scheduler(epoch, lr):
    if epoch < 1000:
        return lr
    elif epoch < 2000:
        return 1e-3
    elif epoch < 4000:
        return 5e-4
    else:
        return 1e-4
        
scheduler_callback = tf.keras.callbacks.LearningRateScheduler(scheduler)

* For those who are excited about `TensorBoard`, you can also add it into a `Callback`

In [ ]:
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir="./tb-logs", update_freq='epoch')
%load_ext tensorboard
%tensorboard --logdir tb-logs

* Now we just need to create a `Context` to elegantly handle the multi-gpu setup and single gpu setup.
* Then we finally choose our favourite optimizer `Adam`
* And we simply take `MSE` as our loss function. You can try a few others like `MAE`. Some studies showed that it might be helpful to choose `MAE`.
* Finally, the most important step is to build an instance of `NIF`, you need to feed three things
  * `cfg_shape_net`: configuration of the shape-net
  * `cfg_parameter_net`: configuration of the parameter-net
  * `mixed_policy`: just a simple string, it can be `mixed_float16`, `float32`, `float16`, `float64`
* Once the NIF class instance it created, let's name it `model_ori` short for **model original** (you can also call it like a hub for the model). However, this is not the model that we really use. Because the way I coded up for hypernetwork. So the real model can be called via `model = model_ori.model()`. **One thing to keep in mind is that when `model` changes as training goes, `model_ori` also changes, you will need this later when you need to just output only shape-net or only parameter-net**
* Then we just compile such `model` and collect our previous `callbacks` (you can just arbitrary choose what kind of `callbacks` you want), then just fit it with our previous datasets and set ups.

In [ ]:
cm = tf.distribute.MirroredStrategy().scope() if enable_multi_gpu else contextlib.nullcontext()
with cm:

    optimizer = nif.optimizers.AdaBeliefOptimizer(lr)
    optimizer.get_gradients = centralized_gradients_for_optimizer(optimizer)

    model_ori = nif.NIF(cfg_shape_net, cfg_parameter_net, mixed_policy)
    model_opt = model_ori.build()

    model_opt.compile(optimizer, loss='mse')

# callbacks = []
# callbacks = [LossAndErrorPrintingCallback(), scheduler_callback]
# callbacks = [tensorboard_callback, ]
callbacks = [tensorboard_callback, LossAndErrorPrintingCallback(), scheduler_callback]
model_opt.fit(train_dataset, epochs=nepoch, batch_size=batch_size,
          shuffle=False, verbose=0, callbacks=callbacks, 
          use_multiprocessing=True)


* Finally, you can just visualize the loss and predictions

In [ ]:
from IPython.display import Image, display

listOfImageNames = ['./loss.png',
                    './vis.png']

for imageName in listOfImageNames:
    display(Image(filename=imageName))

# Get the mapping from parameter to latent representation
* Note it is only part of the ParameterNet

In [ ]:
model_p_to_lr = model_ori.model_p_to_lr()
model_p_to_lr.summary()

* You can also save such model separately
 - note that here you don't have to save the checkpoints for those **derived model**, because anyway they won't be used to further train at all, unless for special purpose

In [ ]:
model_p_to_lr.save('./model_p_to_lr')

# Get the mapping from latent representation to weights and biases

In [ ]:
model_lr_to_w = model_ori.model_lr_to_w()
model_lr_to_w.summary()

# Get ShapeNet conditioned on weights and biases

* there SHOULD NOT be any trainable variables. the below is just to give a sanity-check

In [ ]:
model_x_to_u_given_w = model_ori.model_x_to_u_given_w()
model_x_to_u_given_w.summary()

# Restore model from checkpoint

1. construct a NIF instance (what I previously call *hub of model*) with the **same** configurations for shape net and parameter net, as the one consistent with the checkpoint


In [ ]:
cfg_shape_net = {
    "connectivity": 'full',
    "input_dim": 1,
    "output_dim": 1,
    "units": 30,
    "nlayers": 2,
    "activation": 'swish'
}
cfg_parameter_net = {
    "input_dim": 1,
    "latent_dim": 1,
    "units": 30,
    "nlayers": 2,
    "activation": 'swish',
}
# mixed_policy = 'float32'
new_model_ori = nif.NIF(cfg_shape_net, cfg_parameter_net, mixed_policy)

2. output the real NIF model that needs to be trained, and compile this model
  - note that here we take Adam optimizer, but it won't be used, it is just for compilation purpose.

In [ ]:
new_model = new_model_ori.model()
optimizer = tf.keras.optimizers.legacy.Adam(1e-5)
new_model.compile(optimizer, loss='mse')

3. Now, we load weights from the desired checkpoint directory

In [ ]:
new_model.load_weights("./saved_weights/ckpt-4999/ckpt")
np.testing.assert_allclose(
    model_opt.predict(train_data[:, :2], verbose=0),
    new_model.predict(train_data[:, :2], verbose=0),
    rtol=1e-5, atol=1e-6,
)
print('Checkpoint restore: predictions match.')


# Fine-tuning with L-BFGS
* **it is not used at all in our paper, but just to show that if you want you can**
* early researchers have tried using L-BFGS instead of standard GD/SGD. it does NOT work well in the early stage training deep nets. but in the late stage of training, many works on PINN have reported its usefulness.

In [ ]:
from nif.optimizers import TFPLBFGS

- first we prepare the feature and labels

In [ ]:
data_feature = train_data[:,:2]
data_label = train_data[:,-1:]

- then we use `TFPLBFGS` to wrap around our model, get an instance that can further fine tune the model

In [ ]:
loss_fun = tf.keras.losses.MeanSquaredError()
fine_tuner = TFPLBFGS(new_model, loss_fun, data_feature, data_label, display_epoch=10)

- now we can fine tune the model with `.minimize(rounds, max_iter)`
  - `max_iter` means the we will run LBFGS updates until maximal number of iterations is achieved or error tolerance is achieved
  - here `rounds` means repeating the above process. it can be useful to force running optimization process when error doesn't change even at the level of 1e-15.  

In [ ]:
fine_tuner.minimize(rounds=200, max_iter=1000)
new_model.save_weights("./fine-tuned/ckpt")

history = fine_tuner.history
plt.figure(figsize=(8,2))
plt.semilogy(history['iteration'], history['loss'],'k-o')
plt.ylim([1e-5,1e-2])
plt.xlabel('iteration')
plt.ylabel('loss')
plt.savefig('./fine_tune_loss.png')

## Finally, let's visualize the fine-tuned results

In [ ]:
u_pred = new_model.predict(train_data[:,0:2]).reshape(10,200)
fig,axs=plt.subplots(1,3,figsize=(16,4))
im1=axs[0].contourf(tt, xx, train_data[:,-1].reshape(10,200),vmin=-5,vmax=5,levels=50,cmap='seismic')
plt.colorbar(im1,ax=axs[0])

im2=axs[1].contourf(tt, xx, u_pred,vmin=-5,vmax=5,levels=50,cmap='seismic')
plt.colorbar(im2,ax=axs[1])

im3=axs[2].contourf(tt, xx, (u_pred-train_data[:,-1].reshape(10,200)),vmin=-5,vmax=5,levels=50,cmap='seismic')
plt.colorbar(im3,ax=axs[2])

axs[0].set_xlabel('t')
axs[0].set_ylabel('x')
axs[0].set_title('true')
axs[1].set_title('pred')
axs[2].set_title('error')
plt.savefig('vis_fine_tuned.png')
plt.close()

In [ ]:
display(Image(filename='./vis_fine_tuned.png'))